# 1.0 Import libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
import pandas as pd

# 2.0 Data load

In [6]:
icarda = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/icarda.csv")
ciat = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/compiled_categories_quartiles_modified_for_CIAT-8719_and_CIAT-7714_to_recalculate_ranking.csv")
ilri_gas = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/ilri_gas.csv")
ilri_metdata = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/ilri_metadata.csv")

# 3.0 Functions

In [12]:
# Define the mapping for functional groups
functional_group_mapping = {
    'Grasses': 'Grass',
    'grasses': 'Grass',
    'Shrub': 'Shrub_Trees',
    'Shrub/tree': 'Shrub_Trees',
    'Shrub/Tree': 'Shrub_Trees',
    'Shrub/Trees': 'Shrub_Trees',
    'Shrub_Trees': 'Shrub_Trees',
    'Tree': 'Shrub_Trees',
    'Arbustiva ': 'Shrub_Trees',
    'Herbáceas': 'Herbaceous_legumes',
    'Herbáceas ': 'Herbaceous_legumes',
    'Herbaceous': 'Herbaceous_legumes',
    'Herbaceous legume': 'Herbaceous_legumes',
    'Climber': 'Herbaceous_legumes',
    'Legume': 'Herbaceous_legumes'
}

In [25]:
# Group by subset and id, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

# 4.0 Formating

## 4.1 ICARDA

In [10]:
icarda.head(2)

,id,methane_intensity,tddm,ch4_8h,ch4_24h,functional_group,taxonomic_name,family,Unnamed: 8,Unnamed: 9
0,IGC-2012-78-4-18,14.32,78.53,14.32,17.13,Legume,1(385x 2329)xIGC 2011-61,NaN,NaN,NaN
1,IGC-2012-78-4-18,15.91,76.80,15.91,18.01,Legume,1(385x 2329)xIGC 2011-61,NaN,NaN,NaN


In [8]:
icarda.functional_group.unique()

array(['Legume', 'Grass'], dtype=object)

In [13]:
# Apply the mapping for functional groups
icarda['functional_group'] = icarda['functional_group'].replace(functional_group_mapping)

print('Updated functional group counts for subset_1_information_samples:')
display(icarda['functional_group'].value_counts())

Updated functional group counts for subset_1_information_samples:


,count
functional_group,
Herbaceous_legumes,519
Grass,228


In [20]:
icarda = icarda.drop(columns=['Unnamed: 8', 'Unnamed: 9'])

In [21]:
icarda.head(2)

,id,methane_intensity,tddm,ch4_8h,ch4_24h,functional_group,taxonomic_name,family
0,IGC-2012-78-4-18,14.32,78.53,14.32,17.13,Herbaceous_legumes,1(385x 2329)xIGC 2011-61,NaN
1,IGC-2012-78-4-18,15.91,76.80,15.91,18.01,Herbaceous_legumes,1(385x 2329)xIGC 2011-61,NaN


In [30]:
icarda.columns

Index(['id', 'methane_intensity', 'tddm', 'ch4_8h', 'ch4_24h',
       'functional_group', 'taxonomic_name', 'family'],
      dtype='object')

In [36]:
# Sort columns
icarda_2 = icarda[["id", "taxonomic_name", "family", "functional_group", 'ch4_8h', 'ch4_24h', 'methane_intensity', 'tddm']]
icarda_2 = icarda_2.round(2)

In [40]:
icarda_2.head(2)

,id,taxonomic_name,family,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm
0,IGC-2012-78-4-18,1(385x 2329)xIGC 2011-61,NaN,Herbaceous_legumes,14.32,17.13,14.32,78.53
1,IGC-2012-78-4-18,1(385x 2329)xIGC 2011-61,NaN,Herbaceous_legumes,15.91,18.01,15.91,76.80
